In [1]:
import os
import pandas as pd
import numpy as np
from google.colab import files

# 1. Create the exact VS Code repository folder structure inside Colab
os.makedirs('data/c4_time_series/raw', exist_ok=True)
os.makedirs('data/c4_time_series/interim', exist_ok=True)
os.makedirs('data/c4_time_series/processed', exist_ok=True)

excel_path = 'data/c4_time_series/raw/METEOROLOGICAL DATA.xlsx'

# 2. Upload METEOROLOGICAL DATA.xlsx into data/c4_time_series/raw/
if not os.path.exists(excel_path):
    print("Please upload 'METEOROLOGICAL DATA.xlsx' using the button below:")
    uploaded = files.upload()
    for filename in uploaded.keys():
        os.rename(filename, excel_path)
    print(f"Uploaded successfully to: {excel_path}")
else:
    print(f"File already exists at: {excel_path}")

Please upload 'METEOROLOGICAL DATA.xlsx' using the button below:


Saving METEOROLOGICAL DATA.xlsx to METEOROLOGICAL DATA.xlsx
Uploaded successfully to: data/c4_time_series/raw/METEOROLOGICAL DATA.xlsx


In [2]:
print("Starting Real Mike Flora Weather Cleaning Pipeline...")

# 1. Helper function to unpivot (melt) Monday-Saturday sheets into chronological rows
def extract_sheet(sheet_name, value_col):
    df = pd.read_excel(excel_path, sheet_name=sheet_name)
    df.columns = [str(c).strip() for c in df.iloc[0]]
    df = df[1:].reset_index(drop=True)
    df['YEAR'] = df['YEAR'].ffill().astype(int)
    df['WEEK'] = df['WEEK'].astype(int)

    days = ['MONDAY', 'TUESDAY', 'WEDNESDAY', 'THURSDAY', 'FRIDAY', 'SATURDAY']
    day_offset = {'MONDAY': 0, 'TUESDAY': 1, 'WEDNESDAY': 2, 'THURSDAY': 3, 'FRIDAY': 4, 'SATURDAY': 5}

    melted = df.melt(id_vars=['YEAR', 'WEEK'], value_vars=days, var_name='Day_Name', value_name=value_col)
    melted['Day_Offset'] = melted['Day_Name'].map(day_offset)
    melted = melted.sort_values(['YEAR', 'WEEK', 'Day_Offset']).reset_index(drop=True)
    return melted

# 2. Extract and combine Rainfall, Min Temp, and Max Temp
rain_df = extract_sheet('RAINFALL', 'Rainfall_mm')
min_df = extract_sheet('MINIMUM TEM', 'Min_Temp_C')
max_df = extract_sheet('MAXIMUM TEM', 'Max_Temp_C')

raw = rain_df[['YEAR', 'WEEK', 'Day_Name', 'Day_Offset', 'Rainfall_mm']].copy()
raw['Min_Temp_C'] = min_df['Min_Temp_C']
raw['Max_Temp_C'] = max_df['Max_Temp_C']

# Keep active recorded period (2023 Week 1 to 2026 Week 39)
raw = raw[~((raw['YEAR'] == 2026) & (raw['WEEK'] > 39))].copy()

# 3. Clean text strings ('P/d' for Poya Day, 'H/d' for Holiday) -> convert to NaN
for col in ['Rainfall_mm', 'Min_Temp_C', 'Max_Temp_C']:
    raw[col] = pd.to_numeric(raw[col], errors='coerce')

# Fix clerical typo on 2026 Week 32 Tuesday (34.0 entered in Min_Temp_C while Max_Temp_C was NaN)
typo_mask = (raw['Min_Temp_C'] >= 33.0) & (raw['Max_Temp_C'].isna())
raw.loc[typo_mask, 'Max_Temp_C'] = raw.loc[typo_mask, 'Min_Temp_C']
raw.loc[typo_mask, 'Min_Temp_C'] = np.nan

# 4. Save Cleaned WEEKLY Dataset (ready if sales data tomorrow is weekly)
weekly_df = raw.groupby(['YEAR', 'WEEK'], as_index=False).agg(
    Avg_Max_Temp_C=('Max_Temp_C', 'mean'),
    Avg_Min_Temp_C=('Min_Temp_C', 'mean'),
    Weekly_Rainfall_mm=('Rainfall_mm', 'sum')
)
weekly_df['Avg_Max_Temp_C'] = weekly_df['Avg_Max_Temp_C'].interpolate(method='linear')
weekly_df['Avg_Min_Temp_C'] = weekly_df['Avg_Min_Temp_C'].interpolate(method='linear')
weekly_df['Temp_Diff_C'] = weekly_df['Avg_Max_Temp_C'] - weekly_df['Avg_Min_Temp_C']
weekly_df['Growing_Degree_Days_Weekly'] = (
    np.maximum(0, ((weekly_df['Avg_Max_Temp_C'] + weekly_df['Avg_Min_Temp_C']) / 2.0) - 15.0) * 7.0
)

weekly_output_path = 'data/c4_time_series/interim/cleaned_weather_weekly.csv'
weekly_df.to_csv(weekly_output_path, index=False)

# 5. Build Continuous DAILY Dataset (inserting missing Sundays & interpolating)
raw['Week_Seq'] = (raw['YEAR'] - 2023) * 52 + (raw['WEEK'] - 1)
base_monday = pd.Timestamp('2023-01-02')
raw['Date'] = base_monday + pd.to_timedelta(raw['Week_Seq'] * 7 + raw['Day_Offset'], unit='D')

daily_df = raw.set_index('Date')[['YEAR', 'WEEK', 'Max_Temp_C', 'Min_Temp_C', 'Rainfall_mm']].copy()
full_dates = pd.date_range(start=daily_df.index.min(), end=daily_df.index.max(), freq='D')
daily_df = daily_df.reindex(full_dates)
daily_df.index.name = 'Date'
daily_df = daily_df.reset_index()

# Forward-fill YEAR and WEEK onto Sundays, interpolate temperatures, and zero-fill rainfall
daily_df['YEAR'] = daily_df['YEAR'].ffill().astype(int)
daily_df['WEEK'] = daily_df['WEEK'].ffill().astype(int)
daily_df['Max_Temp_C'] = daily_df['Max_Temp_C'].interpolate(method='linear').bfill().ffill()
daily_df['Min_Temp_C'] = daily_df['Min_Temp_C'].interpolate(method='linear').bfill().ffill()
daily_df['Rainfall_mm'] = daily_df['Rainfall_mm'].fillna(0.0)

# 6. Feature Engineering (Rolling Accumulations & Agronomic Features)
daily_df['Diurnal_Temp_Range'] = daily_df['Max_Temp_C'] - daily_df['Min_Temp_C']
daily_df['Growing_Degree_Days'] = np.maximum(0, ((daily_df['Max_Temp_C'] + daily_df['Min_Temp_C']) / 2.0) - 15.0)
daily_df['7_Day_Avg_Max_Temp'] = daily_df['Max_Temp_C'].rolling(window=7, min_periods=1).mean()
daily_df['7_Day_Total_Rain'] = daily_df['Rainfall_mm'].rolling(window=7, min_periods=1).sum()

daily_output_path = 'data/c4_time_series/interim/cleaned_weather.csv'
daily_df.to_csv(daily_output_path, index=False)

print(f"Saved {len(daily_df)} daily records to {daily_output_path}")
print(f"Saved {len(weekly_df)} weekly records to {weekly_output_path}")
print("\nMissing values check:")
print(daily_df.isnull().sum())
display(daily_df.head(5))

Starting Real Mike Flora Weather Cleaning Pipeline...


/tmp/ipykernel_1511/2107748348.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['YEAR'] = df['YEAR'].ffill().astype(int)
/tmp/ipykernel_1511/2107748348.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['YEAR'] = df['YEAR'].ffill().astype(int)
/tmp/ipykernel_1511/2107748348.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[

Saved 1364 daily records to data/c4_time_series/interim/cleaned_weather.csv
Saved 195 weekly records to data/c4_time_series/interim/cleaned_weather_weekly.csv

Missing values check:
Date                   0
YEAR                   0
WEEK                   0
Max_Temp_C             0
Min_Temp_C             0
Rainfall_mm            0
Diurnal_Temp_Range     0
Growing_Degree_Days    0
7_Day_Avg_Max_Temp     0
7_Day_Total_Rain       0
dtype: int64


,Date,YEAR,WEEK,Max_Temp_C,Min_Temp_C,Rainfall_mm,Diurnal_Temp_Range,Growing_Degree_Days,7_Day_Avg_Max_Temp,7_Day_Total_Rain
0,2023-01-02,2023,1,32.0,31.0,0.0,1.0,16.50,32.000000,0.0
1,2023-01-03,2023,1,32.0,31.0,0.0,1.0,16.50,32.000000,0.0
2,2023-01-04,2023,1,30.0,29.0,0.0,1.0,14.50,31.333333,0.0
3,2023-01-05,2023,1,31.0,30.0,0.0,1.0,15.50,31.250000,0.0
4,2023-01-06,2023,1,32.5,31.0,0.0,1.5,16.75,31.500000,0.0


In [3]:
files.download('data/c4_time_series/interim/cleaned_weather.csv')
files.download('data/c4_time_series/interim/cleaned_weather_weekly.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>